# Phase 3A vs 3B comparative analysis

Tables 1-4 of the Phase 3 guidelines, paired bootstrap intervals, matched slices and the error-review sample.

All numbers come from the exported run artifacts in `results/`. The current cohort is the provisional `gemini-v2` `span_checked` release (322 test question groups, human review pending), so results must be labelled provisional.

In [ ]:
import csv, json
from pathlib import Path
import matplotlib.pyplot as plt

RESULTS = Path.cwd().parent / "results" if Path.cwd().name == "notebooks" else Path("phase_03_multilingual_finetuning/results")
LANGS = ["en", "hi", "mr", "bn", "ta"]

def load_json(name):
    return json.loads((RESULTS / name).read_text(encoding="utf-8"))

def load_csv(name):
    with (RESULTS / name).open(encoding="utf-8", newline="") as stream:
        return list(csv.DictReader(stream))

def show(rows, fields=None, digits=2):
    """Print a plain-text table without extra dependencies."""
    fields = fields or list(rows[0])
    def fmt(value):
        try:
            return f"{float(value):.{digits}f}" if "." in str(value) else str(value)
        except ValueError:
            return str(value)
    cells = [[fmt(row.get(f, "")) for f in fields] for row in rows]
    widths = [max(len(f), *(len(c[i]) for c in cells)) for i, f in enumerate(fields)]
    print("  ".join(f.ljust(w) for f, w in zip(fields, widths)))
    for c in cells:
        print("  ".join(v.ljust(w) for v, w in zip(c, widths)))

def require(*names):
    missing = [n for n in names if not (RESULTS / n).exists()]
    if missing:
        raise FileNotFoundError("Run scripts/export_results.py first. Missing: " + ", ".join(missing))

## Table 1: main results

In [ ]:
require("comparative_results.csv", "performance_gap.csv", "cross_lingual_consistency.csv",
        "confidence_intervals.csv", "computational_statistics.json", "matched_slices.csv", "error_analysis.csv")
show(load_csv("comparative_results.csv"))

## Table 2: English minus Indic F1 gap

In [ ]:
show(load_csv("performance_gap.csv"))

## Table 3: questions answered exactly in n of 5 languages

In [ ]:
joint = load_csv("cross_lingual_consistency.csv")
show(joint)
pairwise = load_json("pairwise_agreement.json")
show([{"pair": f"{r['language_a']}-{r['language_b']}", "3a": r["percent_agree"], "3b": s["percent_agree"]}
      for r, s in zip(pairwise["phase_3a"]["pairwise_agreement"], pairwise["phase_3b"]["pairwise_agreement"])])

## Table 4: computational efficiency

In [ ]:
compute = load_json("computational_statistics.json")
fields = [("Training examples", "training_rows"), ("Tokenized training features", "tokenized_training_features"),
          ("Model parameters", "parameters"), ("Optimizer steps", "optimizer_steps"),
          ("Training time (min)", "training_seconds"), ("GPU", "device_name"), ("Peak GPU memory (GiB)", "peak_gpu_memory_bytes")]
rows = []
for label, key in fields:
    values = {}
    for exp in ("phase_3a", "phase_3b"):
        value = compute[exp]["training"].get(key)
        if key == "training_seconds" and value is not None:
            value = round(value / 60, 1)
        if key == "peak_gpu_memory_bytes" and value is not None:
            value = round(value / 2**30, 2)
        values[exp] = value
    rows.append({"parameter": label, **values})
rows.append({"parameter": "Inference ms per question row",
             **{exp: round(compute[exp]["average_inference_ms_per_question_row"], 2) for exp in ("phase_3a", "phase_3b")}})
show(rows)

## Paired bootstrap (3B minus 3A, 10,000 resamples, 95% CI)

In [ ]:
intervals = load_csv("confidence_intervals.csv")
show([r for r in intervals if r["metric"] == "f1"], ["language", "delta_pp", "lower_pp", "upper_pp", "unit", "units"])
if (RESULTS / "confidence_intervals_context.csv").exists():
    print("\nResampling by shared English passage:")
    show([r for r in load_csv("confidence_intervals_context.csv") if r["metric"] == "f1"],
         ["language", "delta_pp", "lower_pp", "upper_pp", "unit", "units"])

## Matched slices

`question_type` and `numeric_answer` are heuristic labels from the English question and answer. Every current test question has the category Irrigation, and no rows are human-reviewed yet, so those two slices have a single value.

In [ ]:
slices = load_csv("matched_slices.csv")
for dimension in ("question_type", "numeric_answer", "context_tokens", "first_answer_chars"):
    print(f"\n--- {dimension}")
    show([r for r in slices if r["dimension"] == dimension], ["language", "value", "count", "f1_3a", "f1_3b"])

## Error-review worksheet

Reviewers fill `error_category`, `reviewer` and `comments` in `error_analysis.csv`. Some errors may come from translation quality rather than the model.

In [ ]:
errors = load_csv("error_analysis.csv")
coded = [r for r in errors if r["error_category"].strip()]
print(f"{len(errors)} rows ({len(errors) // 2} question-language pairs x 2 models); {len(coded)} coded so far")

## Figures

In [ ]:
for path in sorted((RESULTS / "plots").glob("*.png")):
    fig, ax = plt.subplots(figsize=(8, 4.6))
    ax.imshow(plt.imread(path))
    ax.set_axis_off()
    ax.set_title(path.name)
    plt.show()